In [14]:
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from pyspark.sql import Row

In [16]:
# SpContext.stop() # Remove if you want to re-run this cell
conf = SparkConf().setMaster("local").setAppName("CityDeath") # treat every core of your desktop as an executor
# creating a spark context object named "SpContext"
SpContext = SparkContext(conf = conf)

In [17]:
# Create a SparkSession object named "SpSession" (the config bit is only for Windows!)
# Initialize a SparkSession with the SQL warehouse directory set
SpSession = SparkSession.builder \
    .appName("CityDeath") \
    .config("spark.sql.warehouse.dir", "/content") \
    .getOrCreate()

In [18]:
#Load the CSV file into a RDD
"""--------------------------------------------------------------------------
Load Data
-------------------------------------------------------------------------"""
# Use the raw URL to download the CSV content, not the HTML page
!rm -f 7-city-death-outcome.csv
!wget -O 7-city-death-outcome.csv https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/7-city-death-outcome-1.csv

--2026-03-17 04:44:34--  https://raw.githubusercontent.com/Luo-Innovation-Lab/Big-Data/main/7-city-death-outcome-1.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1872 (1.8K) [text/plain]
Saving to: ‘7-city-death-outcome.csv’

7-city-death-outcom 100%[===================>]   1.83K  --.-KB/s    in 0s      

2026-03-17 04:44:34 (24.0 MB/s) - ‘7-city-death-outcome.csv’ saved [1872/1872]



In [19]:
autoData = SpContext.textFile("7-city-death-outcome.csv")
autoData.cache() # cache the RDD in memory
autoData.take(10)

['X1-death-outcome,X2-doctor,X3-hosptial,X4-income,X5-density',
 '10.89999962,238,822,10.30000019,86',
 '8.300000191,196,867,9.600000381,106',
 '8.399999619,183,718,10.39999962,69',
 '10.10000038,180,668,13,106',
 '11.89999962,176,680,8.899999619,80',
 '8.800000191,168,636,9.100000381,162',
 '9.800000191,161,870,10.39999962,108',
 '8.100000381,154,354,8.399999619,103',
 '11.19999981,153,648,9.899999619,78']

In [20]:
#Remove the first line (contains headers)
dataLines = autoData.filter(lambda x: "X1-death-outcome" not in x)
dataLines.count()

53

In [21]:
# Function to cleanup Data
def CleanupData(inputStr) :
    attList=inputStr.split(",")

    #Create a row with cleaned up and converted data
    values= Row(
        DEATH_RATE=float(attList[0]),\
        DOCTOR_AVAILABILITY=float(attList[1]), \
        HOSPITAL_AVAILABILITY=float(attList[2]),
        INCOME=float(attList[3]),\
        POPULATION_DENSITY=float(attList[4])
    )
    return values

In [22]:
#Run map with the CleanupData method for data cleanup, and create a Row RDD
autoMap = dataLines.map(CleanupData)
autoMap.cache()

#Create a DataFrame with the rwo RDD data.
autoDf = SpSession.createDataFrame(autoMap)

print('====First 10 Row Sample')
autoDf.take(10)

====First 10 Row Sample


[Row(DEATH_RATE=10.89999962, DOCTOR_AVAILABILITY=238.0, HOSPITAL_AVAILABILITY=822.0, INCOME=10.30000019, POPULATION_DENSITY=86.0),
 Row(DEATH_RATE=8.300000191, DOCTOR_AVAILABILITY=196.0, HOSPITAL_AVAILABILITY=867.0, INCOME=9.600000381, POPULATION_DENSITY=106.0),
 Row(DEATH_RATE=8.399999619, DOCTOR_AVAILABILITY=183.0, HOSPITAL_AVAILABILITY=718.0, INCOME=10.39999962, POPULATION_DENSITY=69.0),
 Row(DEATH_RATE=10.10000038, DOCTOR_AVAILABILITY=180.0, HOSPITAL_AVAILABILITY=668.0, INCOME=13.0, POPULATION_DENSITY=106.0),
 Row(DEATH_RATE=11.89999962, DOCTOR_AVAILABILITY=176.0, HOSPITAL_AVAILABILITY=680.0, INCOME=8.899999619, POPULATION_DENSITY=80.0),
 Row(DEATH_RATE=8.800000191, DOCTOR_AVAILABILITY=168.0, HOSPITAL_AVAILABILITY=636.0, INCOME=9.100000381, POPULATION_DENSITY=162.0),
 Row(DEATH_RATE=9.800000191, DOCTOR_AVAILABILITY=161.0, HOSPITAL_AVAILABILITY=870.0, INCOME=10.39999962, POPULATION_DENSITY=108.0),
 Row(DEATH_RATE=8.100000381, DOCTOR_AVAILABILITY=154.0, HOSPITAL_AVAILABILITY=354.0, I

In [23]:
"""--------------------------------------------------------------------------
Perform Descriptive Data Analytics
-------------------------------------------------------------------------"""
print("=== Perform Descriptive Data Analytics ===")
#See descriptive analytics.
#use discribe to show basic statisticas for columns, such as average value, count, std, min, max
autoDf.select("DEATH_RATE", "DOCTOR_AVAILABILITY", "HOSPITAL_AVAILABILITY", "INCOME", "POPULATION_DENSITY").describe().show()



#Find correlation between predictors and target (high correlation columns normally have high predictive power)
print("=== Perform Correlation Analytics ===")
for i in autoDf.columns: #loop through the columns
  if not( isinstance(autoDf.select(i).take(1)[0][0], str)) : # this is for python 3.0+
    print( "Correlation to DEATH_RATE for ", i, autoDf.stat.corr('DEATH_RATE',i)) # correlation between two columns correlation between two columns i and death rate

=== Perform Descriptive Data Analytics ===
+-------+------------------+-------------------+---------------------+------------------+------------------+
|summary|        DEATH_RATE|DOCTOR_AVAILABILITY|HOSPITAL_AVAILABILITY|            INCOME|POPULATION_DENSITY|
+-------+------------------+-------------------+---------------------+------------------+------------------+
|  count|                53|                 53|                   53|                53|                53|
|   mean| 9.305660409716982| 116.09433962264151|    589.7924528301887| 9.435848991943397|110.64150943396227|
| stddev|1.6625299590405331| 37.886604163428046|   332.61830506606486|1.0754416382670777| 47.17972769495497|
|    min|       3.599999905|               60.0|                190.0|       7.199999809|              35.0|
|    max|       12.80000019|              238.0|               1792.0|              13.0|             292.0|
+-------+------------------+-------------------+---------------------+---------------

In [24]:
"""--------------------------------------------------------------------------
Prepare data for ML
-------------------------------------------------------------------------"""
print("=== Prepare data for ML and show the 10 examples labeledPoint ===")
# Transform to a Data Frame for input to Machine Learing
# Drop columns that are not high at correlation (low correlation)
from pyspark.ml.linalg import Vectors
def transformToLabeledPoint(row) : # tranform row to a LabelPoint object
  lp = ( row["DEATH_RATE"], \
  Vectors.dense([
  row["DOCTOR_AVAILABILITY"], \
  row["HOSPITAL_AVAILABILITY"], \
  row["INCOME"],\
  row["POPULATION_DENSITY"]
  ]))
  return lp

autoLp = autoMap.map(transformToLabeledPoint) # use the method to transform to LabalPoint objects
autoDF = SpSession.createDataFrame(autoLp,["label", "features"]) # prepare the data for ML

autoDF.select("label","features").show(10)

=== Prepare data for ML and show the 10 examples labeledPoint ===
+-----------+--------------------+
|      label|            features|
+-----------+--------------------+
|10.89999962|[238.0,822.0,10.3...|
|8.300000191|[196.0,867.0,9.60...|
|8.399999619|[183.0,718.0,10.3...|
|10.10000038|[180.0,668.0,13.0...|
|11.89999962|[176.0,680.0,8.89...|
|8.800000191|[168.0,636.0,9.10...|
|9.800000191|[161.0,870.0,10.3...|
|8.100000381|[154.0,354.0,8.39...|
|11.19999981|[153.0,648.0,9.89...|
|        8.5|[149.0,631.0,10.8...|
+-----------+--------------------+
only showing top 10 rows


In [25]:
"""--------------------------------------------------------------------------
Perform Machine Learning using training data
-------------------------------------------------------------------------"""
print("=== PPerform Machine Learning ===")
# Split into training and testing data
# data frame has a funtion to split the data randomly into traning and test dataset
# return two data sets: one training, one testing
(trainingData, testData) = autoDF.randomSplit([0.9, 0.1])
print("Total training data count: "+str(trainingData.count()))
print("Total testing data count: "+str(testData.count()))


#Build the model on training data
from pyspark.ml.regression import LinearRegression #we use the DataFrame machine learning package
lr = LinearRegression(maxIter=20) # iteration times
lrModel = lr.fit(trainingData) # algorithm.fit() means strat model building by training #lrModel is the trained model object
#Print the metrics
print("=== Learned model parameters ===")
print("Coefficients: " + str(lrModel.coefficients))# print out trained coefficients
print("Intercept: " + str(lrModel.intercept))# print the trained intercept
#Predict on the test data
predictions = lrModel.transform(testData) # use the model to predict the testing data
predictions.select("prediction","label","features").show()

=== PPerform Machine Learning ===
Total training data count: 43
Total testing data count: 10
=== Learned model parameters ===
Coefficients: [0.006616894138599228,0.0009864293840442502,-0.2807135431248069,-0.009921709210360459]
Intercept: 11.76974621270846
+------------------+-----------+--------------------+
|        prediction|      label|            features|
+------------------+-----------+--------------------+
| 9.032800545877485|        5.0|[134.0,525.0,10.3...|
|  9.44273118219214|6.699999809|[109.0,388.0,8.89...|
| 8.998684042295501|8.399999619|[75.0,345.0,9.600...|
|10.711893285957396|8.899999619|[96.0,1792.0,8.89...|
| 9.787474926807946|        9.0|[82.0,347.0,8.800...|
| 9.700503191282708|9.399999619|[82.0,499.0,7.699...|
| 9.425433603493126|10.10000038|[142.0,430.0,10.6...|
| 8.918744749277181|10.10000038|[180.0,668.0,13.0...|
| 8.706402646300553|10.69999981|[82.0,329.0,8.699...|
| 9.954679797440253|12.80000019|[68.0,383.0,7.400...|
+------------------+-----------+----------

In [26]:
"""--------------------------------------------------------------------------
Perform Model Evaluation using test data
-------------------------------------------------------------------------"""
print("=== PPerform Machine Learning ===")
# Find R2 for Linear Regression to evaluate the prediciton performance: R2 value 0-1, the closer to 1 the better
from pyspark.ml.evaluation import RegressionEvaluator
evaluator = RegressionEvaluator(predictionCol="prediction",
                                labelCol="label",metricName="r2") # set evaluation parameters
r2 = evaluator.evaluate(predictions)# start evaluation (compare prediction outcome labels with the real labelled outcomes)
print("The R2 (coefficient of determination) evalution result is: "+str(r2))

=== PPerform Machine Learning ===
The R2 (coefficient of determination) evalution result is: -0.014705679636422886


In [ ]:
# Input your city's attributes here
my_city_doctor_availability = 200.0
my_city_hospital_availability = 700.0
my_city_income = 12.0
my_city_population_density = 90.0

# Create a feature vector for your city
from pyspark.ml.linalg import Vectors
my_city_features = Vectors.dense([
    my_city_doctor_availability,
    my_city_hospital_availability,
    my_city_income,
    my_city_population_density
])

# Create a DataFrame for your city's features (needs a dummy label)
myCityDf = SpSession.createDataFrame([(0.0, my_city_features)], ["label", "features"])

# Make a prediction
my_city_prediction = lrModel.transform(myCityDf)

# Display the predicted death rate
print("Predicted Death Rate for your city:")
my_city_prediction.select("prediction").show()

Predicted MPG for your car:
+------------------+
|        prediction|
+------------------+
|1469.8142412965578|
+------------------+

